# GO:BP ORA Saturation Plot

**Environment:** `clamp-analyses`

Reads the ORA summary produced by `00_bp_saturation_ora_analysis.ipynb` and plots GO:BP coverage (y-axis) against K (number of LVs, x-axis). Each line is one data coverage level; boxplots show distribution over seeds.

In [ ]:
library(here)
library(ggplot2)
library(dplyr)

input_dir <- here("output/03_model_biology/00_archs4/00_pathway_coverage_bp_saturation/00_bp_saturation_ora_analysis")

theme_ng <- function() {
  theme_classic(base_size = 18) +
    theme(
      axis.title        = element_text(size = 24, colour = "black"),
      axis.title.y      = element_text(margin = margin(r = 18)),
      axis.text         = element_text(size = 18, colour = "black"),
      axis.line         = element_line(linewidth = 0.8, colour = "black"),
      axis.ticks        = element_line(linewidth = 0.8, colour = "black"),
      axis.ticks.length = unit(0.22, "cm"),
      panel.grid.major.y = element_line(colour = "#BDBDBD", linewidth = 0.7),
      panel.grid.minor   = element_blank(),
      legend.position    = "right",
      strip.background   = element_blank(),
      strip.text         = element_text(face = "bold", size = 18),
      plot.margin        = margin(t = 16, r = 12, b = 12, l = 42)
    )
}

## Load results

Tries the combined CSV first; falls back to reconstructing from per-model RDS cache.

In [ ]:
csv_path <- file.path(input_dir, "results_CLAMPfull.csv")

if (file.exists(csv_path)) {
  results_df <- read.csv(csv_path, stringsAsFactors = FALSE)
  message("Loaded from CSV: ", nrow(results_df), " rows")
} else {
  rds_files <- sort(list.files(
    file.path(input_dir, "CLAMPfull"),
    pattern = "^rs[0-9]+_k[0-9]+_seed[0-9]+\\.rds$",
    full.names = TRUE
  ))
  if (length(rds_files) == 0) stop("No results found. Run 00_bp_saturation_ora_analysis.ipynb first.")
  message("Reconstructing from ", length(rds_files), " cached RDS files")
  rows <- lapply(rds_files, function(f) {
    m <- regmatches(basename(f), regexec("rs([0-9]+)_k([0-9]+)_seed([0-9]+)", basename(f)))[[1]]
    if (length(m) < 4) return(NULL)
    res <- readRDS(f)
    if (is.null(res$fdr05)) return(NULL)
    data.frame(
      rs_pct            = as.integer(m[2]),
      k_val             = as.integer(m[3]),
      seed              = as.integer(m[4]),
      n_samples         = res$n_samples,
      n_lvs             = res$n_lvs,
      n_top_genes       = res$n_top_genes,
      n_total_bp_fdr05  = res$fdr05$n_total_bp,
      n_sig_bp_fdr05    = res$fdr05$n_sig_bp,
      coverage_bp_fdr05 = res$fdr05$coverage_bp,
      n_total_bp_fdr01  = res$fdr01$n_total_bp,
      n_sig_bp_fdr01    = res$fdr01$n_sig_bp,
      coverage_bp_fdr01 = res$fdr01$coverage_bp,
      stringsAsFactors  = FALSE
    )
  })
  results_df <- do.call(rbind, Filter(Negate(is.null), rows))
  rownames(results_df) <- NULL
  message("Loaded ", nrow(results_df), " rows from cache")
}

k_values     <- sort(unique(results_df$k_val))
avail_levels <- paste0(sort(unique(results_df$rs_pct)), "%")

message("k_values: ", paste(k_values, collapse = ", "))
message("rs_pct levels: ", paste(avail_levels, collapse = ", "))
print(results_df)

## Prepare plot data

In [ ]:
coverage_colors <- c(
  "1%"   = "#2166AC",
  "5%"   = "#4DAC26",
  "10%"  = "#D6604D",
  "25%"  = "#762A83",
  "50%"  = "#E08214",
  "75%"  = "#01665E",
  "100%" = "#000000"
)
coverage_shapes <- c("1%"=16, "5%"=3, "10%"=1, "25%"=17, "50%"=15, "75%"=8, "100%"=18)

plot_df <- results_df %>%
  dplyr::mutate(
    data_label          = factor(paste0(rs_pct, "%"),
                                 levels = paste0(c(1, 5, 10, 25, 50, 75, 100), "%")),
    coverage_bp_fdr05_pct = coverage_bp_fdr05 * 100,
    coverage_bp_fdr01_pct = coverage_bp_fdr01 * 100
  )

make_line_df <- function(y_col) {
  plot_df %>%
    dplyr::group_by(rs_pct, k_val, data_label) %>%
    dplyr::summarise(y = median(.data[[y_col]], na.rm = TRUE), .groups = "drop") %>%
    dplyr::arrange(data_label, k_val)
}

box_width <- diff(range(k_values)) * 0.04

## FDR < 0.05

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 7)

line_df_05 <- make_line_df("coverage_bp_fdr05_pct")

p05 <- ggplot(plot_df, aes(x = k_val, y = coverage_bp_fdr05_pct,
                            colour = data_label, fill = data_label)) +
  geom_boxplot(
    aes(group = interaction(data_label, factor(k_val))),
    alpha = 0.2, outlier.shape = NA, width = box_width,
    position = "identity"
  ) +
  geom_line(
    data = line_df_05,
    aes(x = k_val, y = y, group = data_label),
    linewidth = 1.0
  ) +
  geom_point(aes(shape = data_label), size = 2.5, stroke = 1.0) +
  scale_x_continuous(breaks = k_values) +
  scale_colour_manual(values = coverage_colors, name = "ARCHS4", limits = avail_levels) +
  scale_fill_manual(values   = coverage_colors, name = "ARCHS4", limits = avail_levels) +
  scale_shape_manual(values  = coverage_shapes, name = "ARCHS4", limits = avail_levels) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 0.1, scale = 1),
    breaks = seq(0, 100, by = 2.5),
    limits = c(0, NA),
    expand = expansion(mult = c(0, 0.05))
  ) +
  labs(
    x = "K (number of LVs)",
    y = "GO:BP ORA coverage (FDR < 0.05)"
  ) +
  theme_ng()

p05

## FDR < 0.01

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 7)

line_df_01 <- make_line_df("coverage_bp_fdr01_pct")

p01 <- ggplot(plot_df, aes(x = k_val, y = coverage_bp_fdr01_pct,
                            colour = data_label, fill = data_label)) +
  geom_boxplot(
    aes(group = interaction(data_label, factor(k_val))),
    alpha = 0.2, outlier.shape = NA, width = box_width,
    position = "identity"
  ) +
  geom_line(
    data = line_df_01,
    aes(x = k_val, y = y, group = data_label),
    linewidth = 1.0
  ) +
  geom_point(aes(shape = data_label), size = 2.5, stroke = 1.0) +
  scale_x_continuous(breaks = k_values) +
  scale_colour_manual(values = coverage_colors, name = "ARCHS4", limits = avail_levels) +
  scale_fill_manual(values   = coverage_colors, name = "ARCHS4", limits = avail_levels) +
  scale_shape_manual(values  = coverage_shapes, name = "ARCHS4", limits = avail_levels) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 0.1, scale = 1),
    breaks = seq(0, 100, by = 2.5),
    limits = c(0, NA),
    expand = expansion(mult = c(0, 0.05))
  ) +
  labs(
    x = "K (number of LVs)",
    y = "GO:BP ORA coverage (FDR < 0.01)"
  ) +
  theme_ng()

p01